In [ ]:
#  STEP 1: Install dependencies
!pip install transformers datasets wandb tqdm -q

In [ ]:
# STEP 2: Imports
import random
import numpy as np
import torch
import wandb
from datasets import Dataset, load_dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    GPT2Config,
    Trainer,
    TrainingArguments,
)

In [ ]:
# STEP 3: Fixed config (don't change these)
TOKENIZER_NAME = "CrossBaseArithmetic/crossbase-tokenizer"
DATASET_NAME   = "CrossBaseArithmetic/multi-base-addition-999"
DATASET_CONFIG = None
SEED            = 42
BATCH_SIZE      = 128
LEARNING_RATE   = 1e-3
WARMUP_RATIO    = 0.05
WEIGHT_DECAY    = 1.0
MAX_GRAD_NORM   = 1.0
LOGGING_STEPS   = 100
SAVE_TOTAL_LIMIT = 5
LR_SCHEDULER_TYPE = "cosine"
EVAL_MAX_NEW_TOKENS = 16
MAX_POSITION_EMBEDDINGS = 256
REPORT_TO       = "wandb"

ASSIGNED_HEADS = 4
LAYERS_LIST    = [1]
NUM_EPOCHS     = 10
HF_ORG         = "CrossBaseArithmetic"

# ── CHANGE THIS EACH SESSION ──────────────────────────────────────────────────
# Session 2: run D64  → set DIMS_LIST = [64]
# Session 3: run D128 → set DIMS_LIST = [128]
# Session 4: run D256 → set DIMS_LIST = [256]
# (D32 is always auto-skipped: 32 % 16 != 0)
DIMS_LIST = [32]   # ← only line you change between sessions
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
#  STEP 4: Login to wandb and HuggingFace
wandb.login(key="")  # paste your wandb API key here

from huggingface_hub import login
login(token="")  # paste your HuggingFace token here

In [ ]:
# STEP 5: Reproducibility + device setup
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# STEP 6: Load tokenizer
print(f"Loading tokenizer from '{TOKENIZER_NAME}'...")
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

print(f"Tokenizer vocab size : {len(tokenizer)}")
print(f"  max token ID       : {max(tokenizer.get_vocab().values())}")
print(f"  pad_token_id       : {tokenizer.pad_token_id}")
print(f"  bos_token_id       : {tokenizer.bos_token_id}")
print(f"  eos_token_id       : {tokenizer.eos_token_id}")

In [ ]:
# STEP 7: Load and tokenize dataset
print(f"Loading dataset '{DATASET_NAME}'...")
raw_train = load_dataset(DATASET_NAME, DATASET_CONFIG, split="train")
raw_val   = load_dataset(DATASET_NAME, DATASET_CONFIG, split="validation")

def tokenize(batch):
    texts = [p + a + tokenizer.eos_token for p, a in zip(batch["prompt"], batch["answer"])]
    return tokenizer(texts, truncation=True, max_length=MAX_POSITION_EMBEDDINGS)

tokenized_train = raw_train.map(tokenize, batched=True, remove_columns=raw_train.column_names)
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

print(f"Train examples : {len(raw_train)}")
print(f"Val examples   : {len(raw_val)}")

In [ ]:
# STEP 8: Define inference function + custom Trainer

def run_inference_batch(model, tokenizer, prompts, max_new_tokens=16):
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    inputs.pop("token_type_ids", None)
    prompt_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    completions = []
    for out in output_ids:
        completion = tokenizer.decode(out[prompt_len:], skip_special_tokens=True)
        completions.append(completion.split("\n")[0].strip())
    return completions


class GenerationEvalTrainer(Trainer):
    def __init__(self, *args, eval_batch_size=128, eval_max_new_tokens=16, train_dataset_raw=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.eval_batch_size = eval_batch_size
        self.eval_max_new_tokens = eval_max_new_tokens
        self.train_dataset_raw = train_dataset_raw

    def _run_eval_loop(self, dataset, desc):
        results = []
        for i in tqdm(range(0, len(dataset), self.eval_batch_size), desc=desc):
            batch = dataset[i : i + self.eval_batch_size]
            predictions = run_inference_batch(
                self.model, self.processing_class, batch["prompt"], self.eval_max_new_tokens
            )
            for j, predicted in enumerate(predictions):
                results.append({"predicted": predicted, "answer": batch["answer"][j]})
        return results

    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix="eval"):
        eval_dataset = eval_dataset if eval_dataset is not None else self.eval_dataset
        was_training = self.model.training
        self.model.eval()
        original_padding_side = self.processing_class.padding_side
        self.processing_class.padding_side = "left"

        try:
            metrics = {}

            # Eval split — overall + per base
            eval_results = self._run_eval_loop(eval_dataset, desc="Evaluating (val)")
            eval_accuracy = sum(r["predicted"] == r["answer"] for r in eval_results) / len(eval_results)
            metrics[f"{metric_key_prefix}_accuracy"] = eval_accuracy

            eval_bases = eval_dataset["base"]
            for base in sorted(set(eval_bases)):
                subset = [r for r, b in zip(eval_results, eval_bases) if b == base]
                metrics[f"{metric_key_prefix}_base{base}_accuracy"] = (
                    sum(r["predicted"] == r["answer"] for r in subset) / len(subset) if subset else float("nan")
                )

            # Train split — overall + per base
            if self.train_dataset_raw is not None:
                train_results = self._run_eval_loop(self.train_dataset_raw, desc="Evaluating (train)")
                train_accuracy = sum(r["predicted"] == r["answer"] for r in train_results) / len(train_results)
                metrics["train_accuracy"] = train_accuracy

                train_bases = self.train_dataset_raw["base"]
                for base in sorted(set(train_bases)):
                    subset = [r for r, b in zip(train_results, train_bases) if b == base]
                    metrics[f"train_base{base}_accuracy"] = (
                        sum(r["predicted"] == r["answer"] for r in subset) / len(subset) if subset else float("nan")
                    )

            self.log(metrics)
            self.control = self.callback_handler.on_evaluate(self.args, self.state, self.control, metrics)
            return metrics
        finally:
            self.processing_class.padding_side = original_padding_side
            if was_training:
                self.model.train()

In [ ]:
# STEP 9: Run single model (one dim per session)

import os

EVAL_SAMPLE_SIZE = 10_000
print(f"Sampling {EVAL_SAMPLE_SIZE} rows each for train/val eval probes...")
eval_train_sample = (
    raw_train
    .shuffle(seed=SEED)
    .select(range(min(EVAL_SAMPLE_SIZE, len(raw_train))))
)
eval_val_sample = (
    raw_val
    .shuffle(seed=SEED)
    .select(range(min(EVAL_SAMPLE_SIZE, len(raw_val))))
)
print(f"Train probe : {len(eval_train_sample)} rows")
print(f"Val probe   : {len(eval_val_sample)} rows")

cuda_available = torch.cuda.is_available()
supports_bf16  = cuda_available and torch.cuda.get_device_capability()[0] >= 8
print(f"\nCUDA: {cuda_available} | bf16: {supports_bf16}")
print(f"Running: H={ASSIGNED_HEADS}, layers={LAYERS_LIST}, dims={DIMS_LIST}\n")

for layers in LAYERS_LIST:
    for dim in DIMS_LIST:

        if dim % ASSIGNED_HEADS != 0:
            print(f"Skipping L{layers}_H{ASSIGNED_HEADS}_D{dim} — {dim} not divisible by {ASSIGNED_HEADS}")
            continue

        RUN_NAME   = f"L{layers}_H{ASSIGNED_HEADS}_D{dim}"
        HUB_NAME   = f"{HF_ORG}/arithmetic-model-{RUN_NAME}"
        OUTPUT_DIR = f"./saved_models/{RUN_NAME}"
        os.makedirs(OUTPUT_DIR, exist_ok=True)

        print(f"\n{'='*50}")
        print(f"Starting run: {RUN_NAME}")
        print(f"{'='*50}")

        if wandb.run is not None:
            wandb.finish()

        wandb.init(
            project="CrossBaseArithmetic",
            entity="ahanass2505-pes-university",
            name=RUN_NAME,
            reinit=True,
            config=dict(
                hidden_size=dim, num_layers=layers,
                num_heads=ASSIGNED_HEADS, num_epochs=NUM_EPOCHS,
                batch_size=BATCH_SIZE, lr=LEARNING_RATE,
                weight_decay=WEIGHT_DECAY,
            ),
        )

        config = GPT2Config(
            vocab_size=len(tokenizer),
            n_embd=dim,
            n_inner=4 * dim,
            n_layer=layers,
            n_head=ASSIGNED_HEADS,
            n_positions=MAX_POSITION_EMBEDDINGS,
            pad_token_id=tokenizer.pad_token_id,
            bos_token_id=tokenizer.bos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            resid_pdrop=0.0,
            embd_pdrop=0.0,
            attn_pdrop=0.0,
            summary_first_dropout=0.0,
        )
        model = AutoModelForCausalLM.from_config(config)
        model.to(device)

        print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

        training_args = TrainingArguments(
            output_dir=OUTPUT_DIR,
            num_train_epochs=NUM_EPOCHS,
            per_device_train_batch_size=BATCH_SIZE,
            learning_rate=LEARNING_RATE,
            lr_scheduler_type=LR_SCHEDULER_TYPE,
            warmup_ratio=WARMUP_RATIO,
            weight_decay=WEIGHT_DECAY,
            max_grad_norm=MAX_GRAD_NORM,
            optim="adamw_torch",
            eval_on_start=True,
            eval_strategy="epoch",
            save_strategy="epoch",
            save_total_limit=SAVE_TOTAL_LIMIT,
            load_best_model_at_end=True,
            metric_for_best_model="eval_accuracy",
            greater_is_better=True,
            logging_strategy="steps",
            logging_steps=LOGGING_STEPS,
            push_to_hub=True,
            hub_model_id=HUB_NAME,
            hub_strategy="every_save",
            seed=SEED,
            report_to=REPORT_TO,
            run_name=RUN_NAME,
            bf16=supports_bf16,
        )

        trainer = GenerationEvalTrainer(
            model=model,
            args=training_args,
            train_dataset=tokenized_train,       # full 40M — unchanged
            eval_dataset=eval_val_sample,        # 60k probe
            data_collator=collator,
            processing_class=tokenizer,
            eval_batch_size=BATCH_SIZE,
            eval_max_new_tokens=EVAL_MAX_NEW_TOKENS,
            train_dataset_raw=eval_train_sample, # 60k probe
        )

        trainer.train()
        trainer.push_to_hub()
        tokenizer.push_to_hub(HUB_NAME)

        del model, trainer
        torch.cuda.empty_cache()

        print(f"Done: {RUN_NAME}")

wandb.finish()
print("\nRun complete!")